# 01 — Umami Analytics Ingestion

Validate the Umami Cloud REST API integration:
- Connect with API key
- Fetch website stats, pageviews, metrics, events
- Parse into our `Insights` model
- Save to local state (mock S3)

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

UMAMI_API_KEY = os.getenv('UMAMI_API_KEY')
UMAMI_WEBSITE_ID = os.getenv('UMAMI_WEBSITE_ID', 'e41ae7d9-a536-426d-b40e-f2488b11bf95')

print(f'API Key loaded: {"yes" if UMAMI_API_KEY else "NO — create .env from .env.example"}')
print(f'Website ID: {UMAMI_WEBSITE_ID}')

API Key loaded: yes
Website ID: e41ae7d9-a536-426d-b40e-f2488b11bf95


## 1. Connect to Umami Cloud API

Umami Cloud base URL: `https://api.umami.is/v1`  
Auth: `x-umami-api-key` header  
Rate limit: 50 calls / 15 seconds

In [2]:
from agent.umami_client import UmamiClient, ms_timestamp

umami = UmamiClient(api_key=UMAMI_API_KEY, website_id=UMAMI_WEBSITE_ID)

# Test connection: get active users
active = umami.get_active()
print(f'Active users right now: {active}')

Active users right now: {'visitors': 0}


## 2. Fetch Website Stats (last 7 days)

In [3]:
start = ms_timestamp(days_ago=7)
end = ms_timestamp(days_ago=0)

stats = umami.get_stats(start_at=start, end_at=end)
print('Website stats (7 days):')
for k, v in stats.items():
    if k != 'comparison':
        print(f'  {k}: {v}')

Website stats (7 days):
  pageviews: 58
  visitors: 43
  visits: 43
  bounces: 41
  totaltime: 110


## 3. Top Pages

In [4]:
top_pages = umami.get_metrics(start_at=start, end_at=end, metric_type='path', limit=15)
print('Top pages:')
for page in top_pages:
    print(f'  {page["y"]:>5} visitors — {page["x"]}')

Top pages:
     13 visitors — /
      6 visitors — /amo/13/
      5 visitors — /amo/10
      4 visitors — /blog/
      3 visitors — /de/blog/
      3 visitors — /amo/11
      2 visitors — /de/blog/27/
      2 visitors — /blog/27/
      1 visitors — /quantum/hardware/
      1 visitors — /blog/12/
      1 visitors — /quantum/amo/
      1 visitors — /growth/
      1 visitors — /blog/26/
      1 visitors — /quantum/qml/3/
      1 visitors — /quantum/qml/1/


## 4. Top Referrers

In [5]:
referrers = umami.get_metrics(start_at=start, end_at=end, metric_type='referrer', limit=15)
print('Top referrers:')
for ref in referrers:
    print(f'  {ref["y"]:>5} visitors — {ref["x"]}')

Top referrers:
      2 visitors — fretchen.eu
      1 visitors — go.bsky.app


## 5. Events (tracked funnels)

In [6]:
events = umami.get_metrics(start_at=start, end_at=end, metric_type='event', limit=30)
print('Tracked events:')
for event in events:
    print(f'  {event["y"]:>5}x — {event["x"]}')

Tracked events:
      1x — blog-support-button-hover


## 6. Pageviews Over Time

In [7]:
pageviews = umami.get_pageviews(start_at=start, end_at=end, unit='day')
print('Daily pageviews:')
for pv in pageviews.get('pageviews', []):
    print(f'  {pv["x"][:10]} — {pv["y"]} views')

print('\nDaily sessions:')
for s in pageviews.get('sessions', []):
    print(f'  {s["x"][:10]} — {s["y"]} sessions')

Daily pageviews:
  2026-05-29 — 1 views
  2026-05-30 — 10 views
  2026-05-31 — 1 views
  2026-06-01 — 6 views
  2026-06-02 — 21 views
  2026-06-03 — 7 views
  2026-06-04 — 7 views
  2026-06-05 — 5 views

Daily sessions:
  2026-05-29 — 1 sessions
  2026-05-30 — 10 sessions
  2026-05-31 — 1 sessions
  2026-06-01 — 6 sessions
  2026-06-02 — 8 sessions
  2026-06-03 — 7 sessions
  2026-06-04 — 5 sessions
  2026-06-05 — 5 sessions


## 7. Save to S3 State

In [9]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
store.write('insights.json', insights)

# Verify round-trip
loaded = store.read('insights.json')
print(f'Saved and loaded insights.json ({len(str(loaded))} chars)')
print(f'Top page: {loaded["website_analytics"]["top_pages"][0] if loaded["website_analytics"]["top_pages"] else "none"}')

NameError: name 'insights' is not defined

In [ ]:
umami.close()
print('Done — Umami ingestion validated.')

Done — Umami ingestion validated.
